In [2]:
import os
import pandas as pd
from google.cloud import bigquery
from dotenv import load_dotenv
import xgboost as xgb

load_dotenv() 

project_id = os.getenv("GCP_PROJECT_ID")

if "GOOGLE_APPLICATION_CREDENTIALS" not in os.environ:
    raise ValueError("Error: GCP key not found!")

client = bigquery.Client(project=project_id)
query = f"SELECT * FROM `{project_id}.shop_data.shop_data_view`"
df = client.query(query).to_dataframe(create_bqstorage_client=False)

print("Setting connection with BigQuery and downloading view... ")

print(f"Succes! Data loaded. Table shape: {df.shape}")

display(df.head())


Setting connection with BigQuery and downloading view... 
Succes! Data loaded. Table shape: (95436, 16)


,customer_state,main_seller_state,main_product_category,product_description_lenght,product_photos_qty,product_weight_g,total_payment_value,installments,total_freight,order_status,order_sequence,minutes_for_approval,days_between_orders,estimated_waiting_days,delivery_time_ratio,review_score
0,RS,PR,telefonia,381,1,700,75.06,3,15.56,canceled,<NA>,46877,<NA>,53,NaN,1
1,SP,SP,brinquedos,1493,1,700,142.34,3,12.44,canceled,<NA>,3837,<NA>,51,NaN,5
2,RJ,SP,esporte_lazer,823,1,476,35.61,1,14.11,canceled,<NA>,2058,<NA>,53,0.68,1
3,RJ,RJ,perfumaria,259,1,610,263.93,2,14.03,canceled,<NA>,907,<NA>,53,0.13,1
4,SP,RJ,perfumaria,605,2,100,140.22,2,30.42,canceled,<NA>,816,<NA>,52,0.13,5


In [3]:
text_columns = ['customer_state', 'main_seller_state', 'main_product_category', 'order_status']
df[text_columns] = df[text_columns].astype("category")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95436 entries, 0 to 95435
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   customer_state              95436 non-null  category
 1   main_seller_state           95436 non-null  category
 2   main_product_category       95436 non-null  category
 3   product_description_lenght  95436 non-null  Int64   
 4   product_photos_qty          95436 non-null  Int64   
 5   product_weight_g            95435 non-null  Int64   
 6   total_payment_value         95435 non-null  float64 
 7   installments                95435 non-null  Int64   
 8   total_freight               95436 non-null  float64 
 9   order_status                95436 non-null  category
 10  order_sequence              95003 non-null  Int64   
 11  minutes_for_approval        95423 non-null  Int64   
 12  days_between_orders         3280 non-null   Int64   
 13  estimated_waiting_days     

In [4]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, train_test_split
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

X, y = df.drop('review_score', axis=1), df['review_score'] - 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def fit_and_score(estimator, X_train, X_test, y_train, y_test, weight):
    estimator.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose = False, sample_weight=weight)

    train_score = estimator.score(X_train, y_train)
    test_score = estimator.score(X_test, y_test)

    return estimator, train_score, test_score


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=94)

clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=3, enable_categorical=True)

results = {}

for train, test in cv.split(X_train, y_train):
    X_train_cv = X_train.iloc[train]
    X_test_cv = X_train.iloc[test]
    y_train_cv = y_train.iloc[train]
    y_test_cv = y_train.iloc[test]

    weights_cv = compute_sample_weight(class_weight='balanced', y=y_train_cv)

    est, train_score, test_score = fit_and_score(
        clone(clf), X_train_cv, X_test_cv, y_train_cv, y_test_cv, weights_cv
    )
    results[est] = (train_score, test_score)

for i, (model_instance, scores) in enumerate(results.items()):
    print(f"Iteration {i+1}:")
    print(f" - Training score: {scores[0]:.4f}")
    print(f" - Test score: {scores[1]:.4f}\n")

weights_final = compute_sample_weight(class_weight='balanced', y=y_train)

clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False, sample_weight=weights_final)

booster = clf.get_booster()
print(booster.num_boosted_rounds())

y_pred = clf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nMatrix of Confusions:")
print(confusion_matrix(y_test, y_pred))

Iteration 1:
 - Training score: 0.6270
 - Test score: 0.4122

Iteration 2:
 - Training score: 0.6253
 - Test score: 0.4020

Iteration 3:
 - Training score: 0.6279
 - Test score: 0.3968

Iteration 4:
 - Training score: 0.6273
 - Test score: 0.3987

Iteration 5:
 - Training score: 0.6265
 - Test score: 0.4079

100
Classification Report:
              precision    recall  f1-score   support

         0.0       0.39      0.47      0.42      1879
         1.0       0.05      0.12      0.07       592
         2.0       0.09      0.17      0.12      1511
         3.0       0.23      0.26      0.24      3814
         4.0       0.68      0.48      0.57     11292

    accuracy                           0.40     19088
   macro avg       0.29      0.30      0.28     19088
weighted avg       0.50      0.40      0.44     19088


Matrix of Confusions:
[[ 877  176  236  249  341]
 [ 170   69  106  111  136]
 [ 198  149  259  364  541]
 [ 299  322  684 1004 1505]
 [ 728  773 1643 2675 5473]]
